<a href="https://colab.research.google.com/github/ddickson28/Captain-FPSO-BN/blob/main/TrialBN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install mbnpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.3/105.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 10.0 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: plotly
    Found existing installation: plotly 5.24.1
    Uninstalling plotly-5.24.1:
      Successfully uninstalled plotly-5.24.1
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you 

In [1]:
#Import modules
import numpy as np #importing a module
from mbnpy import variable, cpm, inference #importing relevant code classes from MBNpy
import itertools
from scipy.stats import truncnorm
from math import erf, sqrt

In [4]:
"Building the BN and relationship"
from mbnpy import variable, cpm, inference #importing relevant code classes from MBNpy

#1 Define variables

WeatherDamage = variable.Variable('WeatherDamage', ['0','0.2','0.4','0.6','0.8','1.0']) #Damage taken from BV report, discretised over 6 intervals
LoadingDamage = variable.Variable('LoadingDamage', ['0','0.2','0.4','0.6','0.8','1.0']) #Damage taken from BV report, discretised over 6 intervals
TotalDamage = variable.Variable('TotalDamage', ['0','0.2','0.4','0.6','0.8','1.0']) #Deterministic node, sum of weather and loading
PreviousRepair = variable.Variable('PreviousRepair', ['False','True']) #Repaired or not

RemainingDamage = variable.Variable('RemainingDamage', ['0','0.2','0.4','0.6','0.8','1.0']) # 1 - TotalDamage or// Previous step damage - TotalDamage
RemainingDamage_prev = variable.Variable('RemainingDamage_prev',['0','0.2','0.4','0.6','0.8','1.0'])

CrackLocationInterest = variable.Variable('CrackLocationInterest', ['False', 'True']) # Corrected for consistency with above


#2 Define the cpm of the variables from 1 in order specified above.


cpm_WeatherDamage = cpm.Cpm(
									[WeatherDamage], no_child=1,
									 C=np.array([[0],[1],[2],[3],[4],[5]], dtype=int),
									 #p=np.array([0.017,0.435,0.518,0.03,0.0]) #This needs to be from Excel Norm Distribution
)

cpm_LoadingDamage = cpm.Cpm(
									[LoadingDamage], no_child=1,
									 C=np.array([[0],[1],[2],[3],[4],[5]], dtype=int),
									 p=np.array([0.122,0.677,0.198,0.002,0.0,0.0]) #This needs to be from excel Norm Distribution
)



#First column is total damage, second is loading damage, 3rd is weather damage

cpm_TotalDamage=cpm.Cpm(
		 							[TotalDamage, LoadingDamage, WeatherDamage], no_child=1,
									 C=np.array([
												[0,0,0], #permissible states only e.g. cannot have 0| 0, 1
												[1,0,1],
												[1,1,0], # Corrected typo here
											  [2,2,0],
												[2,1,1],
												[2,0,2],
												[3,3,0],
												[3,2,1],
												[3,1,2],
												[3,0,3],
												[4,4,0],
												[4,3,1],
												[4,2,2],
												[4,1,3],
												[4,0,4],
												[5,5,0],
												[5,0,1],
												[5,4,1],
												[5,3,2],
												[5,2,3],
												[5,1,4],
												[5,0,5], #additional permissible combination inserted
												[5,1,5], #addition WD / LD state 5's not captured above
												[5,2,5],
												[5,3,5],
												[5,4,5],
												[5,5,5],
												[5,5,1],
												[5,5,2],
												[5,5,3],
												[5,5,4],

									 ], dtype=int),
									 p=np.array([1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1])
)


cpm_PreviousRepair = cpm.Cpm(
									[PreviousRepair], no_child=1,
									 C=np.array([[0],[1]], dtype=int),
									 p=np.array([0, 1.0]) #Assume that there hasn't been a repair unless specified otherwise
)

# 6 x 2 x 6 x 6 = 432 combinations reduced to 25 permissible combinations
cpm_RemainingDamage = cpm.Cpm(
									[RemainingDamage,PreviousRepair, RemainingDamage_prev, TotalDamage,], no_child=1,
									C=np.array([[0,0,0,0],
									            [0,0,0,1],
															[0,0,0,2],
															[0,0,0,3],
															[0,0,0,4],
															[0,0,0,5],
															[0,0,5,0],
															[0,0,4,1],
															[0,0,4,2],
									            [0,0,4,3],#
															[0,0,4,4],
															[0,0,4,5],
															[0,0,3,2],
															[0,0,3,3],
															[0,0,3,4],
															[0,0,3,5],
															[0,0,2,3],
															[0,0,2,4],
									            [0,0,2,5],
															[0,0,1,4],
															[0,0,1,5],##
															[0,1,5,5],
															[0,1,4,5],
															[0,1,3,5],
															[0,1,2,5],
															[0,1,1,5],
															[0,1,0,5],
															[1,0,1,0],
															[1,0,2,1],
															[1,0,3,2],
															[1,0,4,3],###
															[1,0,5,4],
															[1,1,5,4],
															[1,1,4,4],
															[1,1,3,4],
															[1,1,2,4],
															[1,1,1,4],
															[1,1,0,4],
															[2,0,2,0],
															[2,0,3,1],
															[2,0,4,2],####
															[2,0,5,3],
															[2,1,5,3],
															[2,1,4,3],
															[2,1,3,3],
															[2,1,2,3],
															[2,1,1,3],
															[2,1,0,3],
															[3,0,3,0],
															[3,0,4,1],
															[3,0,5,2],#####
															[3,1,5,3],
															[3,1,4,3],
															[3,1,3,3],
															[3,1,2,3],
															[3,1,1,3],
															[3,1,0,3],
															[4,0,4,0],
															[4,0,1,1],
															[4,1,5,1],
															[4,1,4,1],######
															[4,1,3,1],
															[4,1,2,1],
															[4,1,1,1],
															[4,1,0,1],
															[5,0,5,0],
															[5,1,5,0],
															[5,1,4,0],
															[5,1,3,0],
															[5,1,2,0],
															[5,1,1,0],#######
															[5,1,0,0],
									  ], dtype=int),
									p=np.array([1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])
)

cpm_CrackLocationInterest = cpm.Cpm(
									[CrackLocationInterest, RemainingDamage], no_child=1,
									 C=np.array([[0,0],
									  				   [0,1],
															 [0,2],
															 [0,3],
															 [0,4],
															 [0,5],
															 [1,0],
															 [1,1],
															 [1,2],
															 [1,3],
															 [1,4],
															 [1,5],

											], dtype=int),
									 p=np.array([0.18, 0.83, 0.99, 1, 1, 1, 0.82, 0.17, 0.01, 0, 0, 0])
)

print(cpm_CrackLocationInterest)

<CPM representing P(CrackLocationInterest | RemainingDamage) at 0x7fbc2bf544a0>
+-------------------------+-------------------+------+
|   CrackLocationInterest [   RemainingDamage ]    p |
+=========================+===================+======+
|                       0 [                 0 ] 0.18 |
+-------------------------+-------------------+------+
|                       0 [                 1 ] 0.83 |
+-------------------------+-------------------+------+
|                       0 [                 2 ] 0.99 |
+-------------------------+-------------------+------+
|                       0 [                 3 ] 1    |
+-------------------------+-------------------+------+
|            :            [         :         ]  :   |
+-------------------------+-------------------+------+
|                       1 [                 3 ] 0    |
+-------------------------+-------------------+------+
|                       1 [                 4 ] 0    |
+-------------------------+-------------